# NEMS price history

Half-hourly USEP, demand, reserve and regulation prices from `data/raw/nems/`.
Fetch or refresh the data with `uv run nems download`.

In [ ]:
import plotly.express as px
import polars as pl

from battery_optimisation.data.nems import load_nems
from battery_optimisation.logs import configure_logging

configure_logging()
df = load_nems()  # one row per 30-min interval, keyed by interval_start (Asia/Singapore)
df.shape

In [ ]:
df.tail()

## Daily averages

Daily means keep the plots light. RUSEP (the uncapped price) only exists from mid-2023, when the temporary price cap started.

In [ ]:
daily = df.group_by_dynamic("interval_start", every="1d").agg(
    pl.col("usep", "rusep", "demand_mw", "regulation", "primary_a", "contingency_a").mean()
)
px.line(daily, x="interval_start", y=["usep", "rusep"], title="Energy price, daily mean (SGD/MWh)")

In [ ]:
px.line(
    daily,
    x="interval_start",
    y=["regulation", "primary_a", "contingency_a"],
    title="Ancillary prices, daily mean (SGD/MWh)",
)

## Last two weeks, half-hourly

In [ ]:
recent = df.filter(
    pl.col("interval_start") >= pl.col("interval_start").max() - pl.duration(days=14)
)
px.line(recent, x="interval_start", y="usep", title="USEP, half-hourly (SGD/MWh)")